# Extract mentions

Joins `risk/risks-v2.tsv` to `risk/studies.tsv` and writes two derived CSVs. The source TSVs are not changed.

- `risk/mentions.csv` — one row per risk coding: `id`, `label`, `ev_id`, `publication_id`
- `risk/ev_titles.csv` — one row per evidence id: `ev_id`, `publication_id`, `title`

`Paper_ID` is the publication id. A blank `Paper_ID` is filled from the single id already used by that title. Run the cells from the top.

In [1]:
from pathlib import Path

import polars as pl

RISKS = Path("risk", "risks-v2.tsv")
STUDIES = Path("risk", "studies.tsv")
MENTIONS = Path("risk", "mentions.csv")
EV_TITLES = Path("risk", "ev_titles.csv")

pl.Config.set_tbl_rows(-1)
pl.Config.set_fmt_str_lengths(500)
pl.Config.set_tbl_width_chars(200)

def show(frame: pl.DataFrame) -> None:
    if frame.height == 0:
        print("none")
    else:
        print(frame)

In [ ]:
risks = pl.read_csv(
    RISKS,
    separator="\t",
    schema_overrides={
        "RISK_ID": pl.Int64,
        "Assotiated Ev_ID": pl.String,
        "Risk": pl.String,
        "Comment": pl.String,
    },
).with_row_index("_order")

print(f"{risks.height} risk rows")

In [ ]:
studies = pl.read_csv(
    STUDIES,
    separator="\t",
    schema_overrides={
        "Title": pl.String,
        "QuickRef": pl.String,
        "Ev_ID": pl.String,
        "Paper_ID": pl.Int64,
        "Cat_ID": pl.Int64,
    },
).with_row_index("_order").with_columns(
    pl.col("Title").str.strip_chars().alias("title"),
    pl.col("Ev_ID").str.strip_chars().alias("ev_id"),
)

print(f"{studies.height} study rows")

In [ ]:
paper_for_title = (
    studies.filter(pl.col("Paper_ID").is_not_null())
    .group_by("title")
    .agg(pl.col("Paper_ID").unique().sort())
    .filter(pl.col("Paper_ID").list.len() == 1)
    .select("title", pl.col("Paper_ID").list.first().alias("filled_paper_id"))
)

studies = studies.join(paper_for_title, on="title", how="left").with_columns(
    pl.coalesce("Paper_ID", "filled_paper_id").alias("publication_id")
)

print("Filled blank Paper_ID")
show(
    studies.filter(pl.col("Paper_ID").is_null()).select(
        "ev_id", "publication_id", "title"
    )
)

if studies.filter(pl.col("publication_id").is_null()).height:
    raise ValueError("A study row has no publication id")

In [ ]:
mentions = (
    risks.with_columns(
        pl.col("Risk").str.strip_chars().alias("label"),
        pl.col("Assotiated Ev_ID").str.strip_chars().alias("ev_id"),
    )
    .join(studies.select("ev_id", "publication_id"), on="ev_id", how="left")
    .sort("_order")
    .select(pl.col("RISK_ID").alias("id"), "label", "ev_id", "publication_id")
)

if mentions.filter(pl.col("publication_id").is_null()).height:
    raise ValueError("A risk row has no publication id")

mentions.write_csv(MENTIONS)
print(f"{mentions.height} mentions -> {MENTIONS}")

In [ ]:
ev_titles = studies.sort("_order").select("ev_id", "publication_id", "title")

ev_titles.write_csv(EV_TITLES)
print(f"{ev_titles.height} evidence ids -> {EV_TITLES}")